In [2]:
%pip install -q evaluate

In [7]:
import pandas as pd
import torch
from pathlib import Path
print("GPU available:", torch.cuda.is_available())

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from datasets import Dataset

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

import transformers
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments, AutoModelForSeq2SeqLM

from evaluate import load as load_metric
import scipy

import numpy as np
import json

import re
import string
import sys

from tqdm.auto import tqdm
tqdm.pandas()


print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("CUDA:", torch.version.cuda)

GPU available: True
Python: 3.13.16
scikit-learn: 1.6.1
NumPy: 2.1.3
SciPy: 1.16.3
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
GPU: Tesla T4
CUDA: 13.0


In [3]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

# Selected languages
langlst = ['ko','ar','te']
lang_map = {
    "ar": "arb_Arab",
    "ko": "kor_Hang",
    "te": "tel_Telu"
}

# Training and Validation data for selected languages
df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#Training and Validation data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]


saved_model_exists = all(
    Path("english-qa", filename).is_file()
    for filename in ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
)
model_source = "english-qa" if saved_model_exists else "bert-base-cased"

qa_tokenizer = AutoTokenizer.from_pretrained(
    model_source, use_fast=True, local_files_only=saved_model_exists
)




README.md:   0%|          | 0.00/9.88k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 6.88MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 1.38MB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 4.80kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15343 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3011 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4 [00:00<?, ? examples/s]

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


nllb_name = "facebook/nllb-200-distilled-600M"

nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
nllb_model = nllb_model.to(device)
nllb_model.eval()






def translate(question, lang):
	source_lang = lang_map[lang]
	nllb_tokenizer.src_lang = source_lang

	inputs = nllb_tokenizer(
		question,
		return_tensors="pt",
		truncation=True,
		max_length=512,
	).to(device)

	with torch.inference_mode():
		translated_tokens = nllb_model.generate(
			**inputs,
			forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"),
		)

	return nllb_tokenizer.batch_decode(
		translated_tokens,
		skip_special_tokens=True
	)[0]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [5]:
from pathlib import Path
import pandas as pd

from tqdm.auto import tqdm
tqdm.pandas()


save_path = Path("translated_dataset.parquet")

if save_path.exists():
    print("Loading previously translated dataset...")
    df_train_filtered = pd.read_parquet(save_path)

else:
    print("Translating questions...")

    df_train_filtered["translated_question"] = df_train_filtered.progress_apply(
        lambda row: translate(
            row["question"],
            row["lang"]
        ),
        axis=1
    )

    df_train_filtered.to_parquet(save_path, index=False)

Loading previously translated dataset...


In [6]:
df_train_filtered

,question,context,lang,answerable,answer_start,answer,answer_inlang,translated_question
0,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,Who is the winner of the Thirty Years' War?
1,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None,Who discovered the X-ray?
2,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None,When was the last time the Olympics were held ...
3,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None,What's the oldest broadcaster in the world?
4,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None,Where is the Palestinian capital?
...,...,...,...,...,...,...,...,...
6330,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,None,When did Somalia make its second inauguration?
6331,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,None,What was the first transportation system in th...
6332,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,None,Who is the leader of Egypt in 2019?
6333,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,None,What is the most densely populated city in Ger...


In [7]:
torch.manual_seed(42)

def prepare_examples(examples):

	tokens = qa_tokenizer(
		examples["translated_question"],
		examples["context"],
		truncation="only_second",
		max_length=384,
		stride=128,
		return_overflowing_tokens=True,
		return_offsets_mapping=True,
		padding="max_length",
	)
	example_numbers = tokens.pop("overflow_to_sample_mapping")
	offsets = tokens.pop("offset_mapping")
	start_positions = []
	end_positions = []

	for i, example_number in enumerate(example_numbers):
		cls_position = tokens["input_ids"][i].index(qa_tokenizer.cls_token_id)
		start_position = cls_position
		end_position = cls_position

		if examples["answerable"][example_number]:
			answer_start = int(examples["answer_start"][example_number])
			answer_end = answer_start + len(examples["answer"][example_number])
			context_tokens = [
				j for j, part in enumerate(tokens.sequence_ids(i)) if part == 1
			]
			if (context_tokens
				and offsets[i][context_tokens[0]][0] <= answer_start
				and offsets[i][context_tokens[-1]][1] >= answer_end):
				answer_tokens = [
					j for j in context_tokens
					if offsets[i][j][0] < answer_end
					and offsets[i][j][1] > answer_start
				]
				if answer_tokens:
					start_position = answer_tokens[0]
					end_position = answer_tokens[-1]

		start_positions.append(start_position)
		end_positions.append(end_position)

	tokens["start_positions"] = start_positions
	tokens["end_positions"] = end_positions
	return tokens

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    model_source,
    local_files_only=saved_model_exists
)

if saved_model_exists:
	training_args = TrainingArguments(output_dir="english-qa", report_to="none")
	trainer = Trainer(model=qa_model, args=training_args)
	print("Loaded saved model. Skipping training.")
else:
	train_examples = Dataset.from_pandas(df_train_filtered, preserve_index=False)
	train_features = train_examples.map(
		prepare_examples, batched=True,
		remove_columns=train_examples.column_names,
	)

	training_args = TrainingArguments(
		output_dir="english-qa",
		num_train_epochs=3,
		per_device_train_batch_size=8,
		# per_device_train_batch_size=2,
		# gradient_accumulation_steps=4,
		fp16=True,
		learning_rate=3e-5,
		seed=42,
		save_strategy="no",
		report_to="none",
		full_determinism=True,
	)

	trainer = Trainer(
		model=qa_model,
		args=training_args,
		train_dataset=train_features
	)

	trainer.train()

	trainer.save_model("english-qa")
	qa_tokenizer.save_pretrained("english-qa")
	print("Saved final model and tokenizer.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded saved model. Skipping training.


In [8]:

save_path = Path("translated_validation_dataset.parquet")

if save_path.exists():
    print("Loading previously translated dataset...")
    df_validation_filtered = pd.read_parquet(save_path)

else:
    print("Translating questions...")

    df_validation_filtered["translated_question"] = df_validation_filtered.progress_apply(
        lambda row: translate(
            row["question"],
            row["lang"]
        ),
        axis=1
    )

    df_validation_filtered.to_parquet(save_path, index=False)


validation_examples = Dataset.from_pandas(
    df_validation_filtered, preserve_index=False
)



def prepare_validation(examples):
    tokens = qa_tokenizer(
        examples["translated_question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Local example indices within this preprocessing batch.
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers

    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        # Only context tokens can belong to an extracted answer.
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    return tokens

# One batch makes example_index refer to the whole validation dataset.
validation_features = validation_examples.map(
    prepare_validation,
    batched=True,
    batch_size=len(validation_examples),
    remove_columns=validation_examples.column_names,
)

# Metadata is needed for decoding, but must not be passed to the model.
model_inputs = validation_features.remove_columns(
    ["offset_mapping", "example_index"]
)
start_logits, end_logits = trainer.predict(model_inputs).predictions

n = len(validation_examples)
null_scores = np.full(n, np.inf)
span_scores = np.full(n, -np.inf)
best_answers = [""] * n

# A fixed decoding choice: number of WordPiece tokens, not words.
MAX_ANSWER_LENGTH = 30

for feature_index, feature in enumerate(validation_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]

    cls_index = feature["input_ids"].index(qa_tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index],
        float(starts[cls_index] + ends[cls_index]),
    )

    # Search all valid context spans up to MAX_ANSWER_LENGTH.
    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue

        for end in range(
            start, min(start + MAX_ANSWER_LENGTH, len(offsets))
        ):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue

            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                context = validation_examples[example_index]["context"]
                best_answers[example_index] = context[
                    start_offset[0]:end_offset[1]
                ]

assert np.isfinite(null_scores).all()
assert np.isfinite(span_scores).all()

no_answer_scores = null_scores - span_scores

Loading previously translated dataset...


Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [9]:
scores = np.asarray(no_answer_scores)
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)

thresholds = []

# First, test a cutoff that predicts no answer for everyone.
threshold_below_all_scores = scores.min() - 1
thresholds.append(threshold_below_all_scores)

# np.unique returns the distinct scores in ascending order.
unique_scores = np.unique(scores)

# Then test each score as a cutoff.
for score in unique_scores:
    thresholds.append(score)

best_threshold = thresholds[0]
best_macro_f1 = -1

for threshold in thresholds:
    predicted_answerable = scores <= threshold

    macro_f1 = f1_score(
        gold_answerable,
        predicted_answerable,
        labels=[False, True],
        average="macro",
        zero_division=0,
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

NO_ANSWER_THRESHOLD = best_threshold

print("Threshold:", NO_ANSWER_THRESHOLD)
print("Validation macro-F1:", best_macro_f1)

Threshold: -1.363856554031372
Validation macro-F1: 0.8613019867372944


In [10]:
predicted_answers = [
    answer if score <= NO_ANSWER_THRESHOLD else ""
    for answer, score in zip(best_answers, scores)
]

In [11]:


# Lab 6 recommends squad_v2 for the project's unanswerable examples.
squad_metric = load_metric("squad_v2")

evaluation = df_validation_filtered.copy()
evaluation["prediction"] = predicted_answers
evaluation["pred_answerable"] = evaluation["prediction"] != ""

results = []
for language in ["overall", "ar", "ko", "te"]:
    group = evaluation if language == "overall" else evaluation[evaluation["lang"] == language]
    formatted_predictions = []
    gold = []

    for example_id, example in group.iterrows():
        formatted_predictions.append({
            "id": str(example_id),
            "prediction_text": example["prediction"],
            # Required field: encode the already-made abstention decision.
            # This is a binary indicator, not a calibrated probability.
            "no_answer_probability": float(not example["pred_answerable"]),
        })
        gold.append({
            "id": str(example_id),
            "answers": {
                "text": [example["answer"]] if example["answerable"] else [],
                "answer_start": [int(example["answer_start"])] if example["answerable"] else [],
            },
        })

    squad_results = squad_metric.compute(
        references=gold, predictions=formatted_predictions
    )
    report = classification_report(
        group["answerable"], group["pred_answerable"],
        labels=[False, True], target_names=["unanswerable", "answerable"],
        output_dict=True, zero_division=0,
    )

    row = {
        "language": language,
        "N": len(group),
        "SQuAD_v2_EM": squad_results["exact"],
        "SQuAD_v2_token_F1": squad_results["f1"],
        "answerability_macro_F1": 100 * report["macro avg"]["f1-score"],
    }
    for label in ["unanswerable", "answerable"]:
        row[f"{label}_P"] = 100 * report[label]["precision"]
        row[f"{label}_R"] = 100 * report[label]["recall"]
        row[f"{label}_F1"] = 100 * report[label]["f1-score"]
    results.append(row)

results_table = pd.DataFrame(results)
display(results_table.round(2))

,language,N,SQuAD_v2_EM,SQuAD_v2_token_F1,answerability_macro_F1,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1
0,overall,1155,57.23,66.65,86.13,84.33,68.90,75.84,95.00,97.88,96.42
1,ar,415,60.48,69.44,93.61,85.71,92.31,88.89,98.89,97.80,98.34
2,ko,356,57.30,68.62,86.39,66.67,84.21,74.42,99.10,97.63,98.36
3,te,384,53.65,61.83,79.39,90.74,52.69,66.67,86.67,98.28,92.11


In [12]:
def normalize_answer(text):
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

examples = evaluation.copy()
examples["gold_answer"] = examples["answer"].where(
    examples["answerable"], ""
)

errors = examples[
    examples["prediction"].map(normalize_answer)
    != examples["gold_answer"].map(normalize_answer)
]

selected_errors = errors.groupby("lang", sort=False).head(4)

print("Selected errors:", len(selected_errors))
print(selected_errors["lang"].value_counts())

for index, row in selected_errors.iterrows():
    print(f"\nExample {index} ({row['lang']})")
    print("Question:", row["question"])
    print("Context:", row["context"])
    print("Gold answer:", repr(row["gold_answer"]))
    print("Prediction:", repr(row["prediction"]))

Selected errors: 12
lang
te    4
ko    4
ar    4
Name: count, dtype: int64

Example 2 (te)
Question: కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?
Context: Since it became widespread in the 19th century, cholera has killed tens of millions of people. In Russia alone, between 1847 and 1851, more than one million people perished of the disease. It killed 150,000 Americans during the second pandemic. Between 1900 and 1920, perhaps eight million people died of cholera in India. Cholera became the first reportable disease in the United States due to the significant effects it had on health. John Snow, in England, was the first to identify the importance of contaminated water as its cause in 1854. Cholera is now no longer considered a pressing health threat in Europe and North America due to filtering and chlorination of water supplies, but still heavily affects populations in developing countries.
Gold answer: 'England'
Prediction: 'India'

Example 3 (te)
Question: మొదటి ప్రపంచ యుద్ధం ఎప్పుడు

In [13]:
from tqdm.auto import tqdm
tqdm.pandas()

records = json.loads(Path("challenge.json").read_text(encoding="utf-8"))
challenge = pd.DataFrame(records)

challenge["translated_question"] = challenge.progress_apply(
        lambda row: translate(
            row["question"],
            row["lang"]
        ),
        axis=1
    )



challenge_examples = Dataset.from_pandas(
    challenge, preserve_index=False
)


  0%|          | 0/30 [00:00<?, ?it/s]

In [14]:
def prepare_challenge(batch):
    tokens = qa_tokenizer(
        batch["translated_question"], batch["context"],
        truncation="only_second", max_length=384, stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    tokens["example_index"] = tokens.pop("overflow_to_sample_mapping")
    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]
    return tokens

challenge_dataset = Dataset.from_list(records)
challenge_dataset = challenge_dataset.map(
    lambda row: {
        "translated_question": translate(
            row["question"],
            row["lang"]
        )
    },
    desc="Translating questions"
)

qa_features = challenge_dataset.map(
    prepare_challenge, batched=True, batch_size=len(records),
    remove_columns=challenge_dataset.column_names,
)
model_inputs = qa_features.remove_columns(["offset_mapping", "example_index"])
start_logits, end_logits = trainer.predict(model_inputs).predictions

Translating questions:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [15]:
null_scores = np.full(len(records), np.inf)
span_scores = np.full(len(records), -np.inf)
best_answers = [""] * len(records)
for feature_index, feature in enumerate(qa_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]
    cls_index = feature["input_ids"].index(qa_tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index], float(starts[cls_index] + ends[cls_index])
    )
    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue
        for end in range(start, min(start + MAX_ANSWER_LENGTH, len(offsets))):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue
            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                best_answers[example_index] = records[example_index]["context"][start_offset[0]:end_offset[1]]

assert np.isfinite(null_scores).all()
assert np.isfinite(span_scores).all()
no_answer_scores = null_scores - span_scores


answer_predictions = {}
answerability_predictions = {}

answer_predictions["English QA"] = [
    answer if score <= NO_ANSWER_THRESHOLD else ""
    for answer, score in zip(best_answers, no_answer_scores)
]
answerability_predictions["English QA"] = [
    answer != "" for answer in answer_predictions["English QA"]
]


In [16]:
answerability_rows = []
for language in ["all", "ar", "ko", "te"]:
    mask = np.ones(len(challenge), dtype=bool)
    if language != "all":
        mask = challenge["lang"] == language
    gold = challenge.loc[mask, "answerable"]
    for name, predictions in answerability_predictions.items():
        precision, recall, f1, _ = precision_recall_fscore_support(
            gold, np.asarray(predictions)[mask],
            labels=[False, True], zero_division=0,
        )
        answerability_rows.append({
            "model": name,
            "language": language,
            "n": len(gold),
            "unanswerable_P": precision[0],
            "unanswerable_R": recall[0],
            "unanswerable_F1": f1[0],
            "answerable_P": precision[1],
            "answerable_R": recall[1],
            "answerable_F1": f1[1],
            "macro_F1": f1.mean(),
        })

answerability_results = pd.DataFrame(answerability_rows)
display(answerability_results.round(4))

,model,language,n,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,macro_F1
0,English QA,all,30,0.0,0.0,0.0,0.5,1.0,0.6667,0.3333
1,English QA,ar,10,0.0,0.0,0.0,0.5,1.0,0.6667,0.3333
2,English QA,ko,10,0.0,0.0,0.0,0.5,1.0,0.6667,0.3333
3,English QA,te,10,0.0,0.0,0.0,0.5,1.0,0.6667,0.3333


In [18]:
#squad_metric = evaluate.load("squad_v2")
qa_eng_predictions = []
qa_eng_gold_answers = []
for i, row in enumerate(records):
    prediction = {}
    prediction["id"] = row["id"]
    prediction["prediction_text"] = answer_predictions["English QA"][i]
    # The frozen threshold has already determined whether to abstain.
    prediction["no_answer_probability"] = 0.0
    qa_eng_predictions.append(prediction)

    answer = {}
    if row["answerable"]:
        answer["text"] = [row["answer"]]
        answer["answer_start"] = [row["answer_start"]]
    else:
        answer["text"] = []
        answer["answer_start"] = []

    gold_answer = {}
    gold_answer["id"] = row["id"]
    gold_answer["answers"] = answer
    qa_eng_gold_answers.append(gold_answer)

qa_eng_rows = []
for language in ["all", "ar", "ko", "te"]:
    predictions = []
    gold_answers = []
    for i, row in enumerate(records):
        if language != "all" and row["lang"] != language:
            continue
        predictions.append(qa_eng_predictions[i])
        gold_answers.append(qa_eng_gold_answers[i])
    scores = squad_metric.compute(predictions=predictions, references=gold_answers)
    qa_eng_rows.append({
        "model": "English QA",
        "language": language,
        "n": len(predictions),
        "F1": scores["f1"] / 100,
        "EM": scores["exact"] / 100,
    })

qa_eng_results = pd.DataFrame(qa_eng_rows)
display(qa_eng_results.round(4))

,model,language,n,F1,EM
0,English QA,all,30,0.4015,0.3333
1,English QA,ar,10,0.5000,0.5000
2,English QA,ko,10,0.4167,0.3000
3,English QA,te,10,0.2879,0.2000


In [29]:
correct_predictions = pd.DataFrame(index=challenge.index)
eng_correct = []
for i in range(len(records)):
    prediction = qa_eng_predictions[i]
    gold_answer = qa_eng_gold_answers[i]
    score = squad_metric.compute(predictions=[prediction], references=[gold_answer])
    eng_correct.append(score["exact"] == 100)
correct_predictions["English QA"] = eng_correct

In [30]:
for name in correct_predictions:
    correct = correct_predictions[name]

    correct_answerable = correct[challenge["answerable"]].sum()
    correct_unanswerable = correct[~challenge["answerable"]].sum()

    correct_pairs = 0
    total_pairs = 0
    for pair_id, pair in challenge.groupby("pair_id", sort=False):
        if correct[pair.index].all():
            correct_pairs += 1
        total_pairs += 1

    print("\nMODEL:", name)
    print("Correct answerable:", correct_answerable, "/", challenge["answerable"].sum())
    print("Correct unanswerable:", correct_unanswerable, "/", (~challenge["answerable"]).sum())
    print("Correct pairs:", correct_pairs, "/", total_pairs)


MODEL: English QA
Correct answerable: 10 / 15
Correct unanswerable: 0 / 15
Correct pairs: 0 / 15


In [31]:
paired_rows = []
for language in ["all", "ar", "ko", "te"]:
    group = challenge
    if language != "all":
        group = challenge[challenge["lang"] == language]
    n_answerable = group["answerable"].sum()
    n_unanswerable = (~group["answerable"]).sum()

    for name, predictions in answerability_predictions.items():
        predicted_answerable = pd.Series(predictions, index=challenge.index)
        correct = predicted_answerable.loc[group.index] == group["answerable"]
        correct_answerable = correct[group["answerable"]].sum()
        correct_unanswerable = correct[~group["answerable"]].sum()

        correct_pairs = 0
        total_pairs = 0
        for pair_id, pair in group.groupby("pair_id", sort=False):
            if correct.loc[pair.index].all():
                correct_pairs += 1
            total_pairs += 1

        row = {
            "model": name,
            "language": language,
            "n_answerable": n_answerable,
            "n_unanswerable": n_unanswerable,
            "correct_answerable": correct_answerable,
            "correct_unanswerable": correct_unanswerable,
            "answerable_R": correct_answerable / n_answerable if n_answerable else 0.0,
            "unanswerable_R": correct_unanswerable / n_unanswerable if n_unanswerable else 0.0,
            "answerability_correct_pairs": correct_pairs,
            "n_pairs": total_pairs,
            "exact_answerable": None,
            "exact_match_pairs": None,
        }

        if name in answer_predictions:
            exact_correct = correct_predictions.loc[group.index, name]
            row["exact_answerable"] = exact_correct[group["answerable"]].sum()
            exact_pairs = 0
            for pair_id, pair in group.groupby("pair_id", sort=False):
                if exact_correct.loc[pair.index].all():
                    exact_pairs += 1
            row["exact_match_pairs"] = exact_pairs
        paired_rows.append(row)

paired_results = pd.DataFrame(paired_rows)
display(paired_results.round(4))

,model,language,n_answerable,n_unanswerable,correct_answerable,correct_unanswerable,answerable_R,unanswerable_R,answerability_correct_pairs,n_pairs,exact_answerable,exact_match_pairs
0,English QA,all,15,15,15,0,1.0,0.0,0,15,10,0
1,English QA,ar,5,5,5,0,1.0,0.0,0,5,5,0
2,English QA,ko,5,5,5,0,1.0,0.0,0,5,3,0
3,English QA,te,5,5,5,0,1.0,0.0,0,5,2,0
